In [3]:
# ! python -m pip install pypdf

In [5]:
from pathlib import Path
import json

from pypdf import PdfReader

In [9]:
INPUT_DIR = Path("../shared/input/financiera/contratos")
OUTPUT_DIR = Path("../shared/output/financiera/textos")

CONTRATOS_DIR = OUTPUT_DIR / "contratos"
PAGOS_DIR = OUTPUT_DIR / "pagos"

MIN_CHARS = 30

def extraer_texto_pdf(pdf_path):
    try:
        reader = PdfReader(pdf_path)

        paginas = []

        for numero, page in enumerate(reader.pages, start=1):
            texto = page.extract_text() or ""
            texto = texto.strip()

            paginas.append({
                "pagina": numero,
                "texto": texto
            })

        texto_completo = "\n\n".join(
            pagina["texto"]
            for pagina in paginas
            if pagina["texto"]
        )

        return {
            "ok": True,
            "paginas": paginas,
            "texto": texto_completo,
            "num_paginas": len(reader.pages)
        }

    except Exception as error:
        return {
            "ok": False,
            "error": str(error),
            "paginas": [],
            "texto": "",
            "num_paginas": 0
        }


def guardar_texto(path, texto):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(texto, encoding="utf-8")


def procesar_pdf(pdf_path, output_path, tipo, contrato):
    resultado = extraer_texto_pdf(pdf_path)

    texto = resultado["texto"]

    requiere_ocr = (
        not resultado["ok"]
        or len(texto.strip()) < MIN_CHARS
    )

    if resultado["ok"] and texto:
        guardar_texto(output_path, texto)

    return {
        "contrato": contrato,
        "tipo": tipo,
        "archivo": pdf_path.name,
        "origen": str(pdf_path),
        "salida": str(output_path) if texto else None,
        "paginas": resultado["num_paginas"],
        "caracteres": len(texto),
        "texto_recuperable": not requiere_ocr,
        "requiere_ocr": requiere_ocr,
        "error": resultado.get("error")
    }

In [ ]:
CONTRATOS_DIR.mkdir(parents=True, exist_ok=True)
PAGOS_DIR.mkdir(parents=True, exist_ok=True)

indice = []

for contrato_dir in sorted(INPUT_DIR.iterdir()):
    if not contrato_dir.is_dir():
        continue

    contrato = contrato_dir.name

    print(f"\nContrato {contrato}")

    # -----------------------------------------
    # Contrato
    # -----------------------------------------

    contratos_pdf = sorted(
        contrato_dir.glob("contrato-*.pdf")
    )

    for pdf_path in contratos_pdf:
        output_path = CONTRATOS_DIR / f"{contrato}.txt"

        resultado = procesar_pdf(
            pdf_path=pdf_path,
            output_path=output_path,
            tipo="contrato",
            contrato=contrato
        )

        indice.append(resultado)

        estado = (
            "TEXTO"
            if not resultado["requiere_ocr"]
            else "OCR"
        )

        print(
            f"  [{estado}] {pdf_path.name} "
            f"({resultado['caracteres']} caracteres)"
        )

    # -----------------------------------------
    # Pagos
    # -----------------------------------------

    pagos_dir = contrato_dir / "pagos"

    if not pagos_dir.exists():
        continue

    for pdf_path in sorted(pagos_dir.glob("*.pdf")):
        output_path = (
            PAGOS_DIR
            / contrato
            / f"{pdf_path.stem}.txt"
        )

        resultado = procesar_pdf(
            pdf_path=pdf_path,
            output_path=output_path,
            tipo="pago",
            contrato=contrato
        )

        indice.append(resultado)

        estado = (
            "TEXTO"
            if not resultado["requiere_ocr"]
            else "OCR"
        )

        print(
            f"  [{estado}] {pdf_path.name} "
            f"({resultado['caracteres']} caracteres)"
        )

# ---------------------------------------------
# Índice general
# ---------------------------------------------

indice_path = OUTPUT_DIR / "indice.json"

indice_path.write_text(
    json.dumps(
        indice,
        indent=4,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

# ---------------------------------------------
# Resumen
# ---------------------------------------------

total = len(indice)

texto = sum(
    1
    for item in indice
    if not item["requiere_ocr"]
)

ocr = sum(
    1
    for item in indice
    if item["requiere_ocr"]
)

errores = sum(
    1
    for item in indice
    if item["error"]
)

print()
print("=" * 60)
print(f"PDF procesados : {total}")
print(f"Con texto      : {texto}")
print(f"Requieren OCR  : {ocr}")
print(f"Errores        : {errores}")
print("=" * 60)
print(f"Índice: {indice_path}")


Contrato 1004
  [TEXTO] contrato-1787615272263.pdf (1721 caracteres)
  [TEXTO] pago-comprobante-1-1787617197485.pdf (978 caracteres)
  [TEXTO] pago-comprobante-2-1788222863236.pdf (998 caracteres)
  [TEXTO] pago-comprobante-3-1788827300843.pdf (1005 caracteres)
  [TEXTO] pago-comprobante-4-1789430969704.pdf (1007 caracteres)

Contrato 1008
  [TEXTO] contrato-1788030441032.pdf (2545 caracteres)
  [TEXTO] pago-comprobante-1-1788030982833.pdf (984 caracteres)
  [TEXTO] pago-comprobante-2-1788736030813.pdf (1008 caracteres)
  [TEXTO] pago-comprobante-3-1789317888189.pdf (1014 caracteres)

Contrato 1009
  [TEXTO] contrato-1788030743683.pdf (1721 caracteres)
  [TEXTO] pago-comprobante-1-1788031465904.pdf (984 caracteres)
  [TEXTO] pago-comprobante-2-1788547025793.pdf (1009 caracteres)
  [TEXTO] pago-comprobante-3-1789250433247.pdf (1001 caracteres)

Contrato 1011
  [TEXTO] contrato-1788053159022.pdf (1743 caracteres)
  [TEXTO] pago-comprobante-1-1788053794326.pdf (1002 caracteres)
  [TEXTO]